##### Implementation of Apriori Algorithms

In [1]:
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import  apriori, association_rules

In [2]:
#loading and exploring the data
data= pd.read_excel('Online Retail.xlsx')
data.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
data.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

In [4]:
#Exploring the different regions of transation
data.Country.unique()

<StringArray>
[      'United Kingdom',               'France',            'Australia',
          'Netherlands',              'Germany',               'Norway',
                 'EIRE',          'Switzerland',                'Spain',
               'Poland',             'Portugal',                'Italy',
              'Belgium',            'Lithuania',                'Japan',
              'Iceland',      'Channel Islands',              'Denmark',
               'Cyprus',               'Sweden',              'Austria',
               'Israel',              'Finland',              'Bahrain',
               'Greece',            'Hong Kong',            'Singapore',
              'Lebanon', 'United Arab Emirates',         'Saudi Arabia',
       'Czech Republic',               'Canada',          'Unspecified',
               'Brazil',                  'USA',   'European Community',
                'Malta',                  'RSA']
Length: 38, dtype: str

In [5]:
# Cleaing the data
#stripping the extra space in the description
data['Description']= data['Description'].str.strip()

#dropping the rows without any invoice number
data.dropna(axis=0, subset=['InvoiceNo'], inplace=True)
data['InvoiceNo'] = data['InvoiceNo'].astype('str')

#Dropping all transaction which were done on credit
data= data[~data['InvoiceNo'].str.contains('C')]

##### Splitting the data according to the region of transaction


##### Transations done in France 

In [6]:
basket_France=(data[data['Country'] == "France"].
               groupby(['InvoiceNo', 'Description'])['Quantity'].
               sum().unstack().reset_index().fillna(0).
               set_index('InvoiceNo'))

##### Transations done in United Kingdom

In [7]:
basket_UK=(data[data['Country'] == "United Kingdom"].
               groupby(['InvoiceNo', 'Description'])['Quantity'].
               sum().unstack().reset_index().fillna(0).
               set_index('InvoiceNo'))

##### Transations done in Portugal

In [8]:
basket_Port=(data[data['Country'] == "Portugal"].
               groupby(['InvoiceNo', 'Description'])['Quantity'].
               sum().unstack().reset_index().fillna(0).
               set_index('InvoiceNo'))

##### Transations done in Sweden

In [9]:
basket_Sweden=(data[data['Country'] == "Sweden"].
               groupby(['InvoiceNo', 'Description'])['Quantity'].
               sum().unstack().reset_index().fillna(0).
               set_index('InvoiceNo'))

In [10]:
# Hot encoding the data
def hot_encode(x):
    if(x <=0):
        return 0
    if(x >=1):
        return 1
#Encoding the datasets
basket_encoded = basket_France.map(hot_encode)
basket_France = basket_encoded

basket_encoded = basket_UK.map(hot_encode)
basket_UK = basket_encoded

basket_encoded = basket_Port.map(hot_encode)
basket_Port = basket_encoded

basket_encoded = basket_Sweden.map(hot_encode)
basket_Sweden = basket_encoded


    

##### Building the models and analyzing the result

In [11]:
#Build the model
freq_items = apriori(basket_France,min_support=0.05, 
                     use_colnames = True)
#Collecting the inferred rules in a dataframe

rules = association_rules(freq_items, metric="lift",
                          min_threshold =1)
rules = rules.sort_values(['confidence', 'lift'], 
                          ascending=[False, False])
print(rules.head())

D:\Python 311\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


                                           antecedents  \
44             frozenset({JUMBO BAG WOODLAND ANIMALS})   
258  frozenset({RED TOADSTOOL LED NIGHT LIGHT, PLAS...   
270  frozenset({PLASTERS IN TIN WOODLAND ANIMALS, R...   
302  frozenset({SET/6 RED SPOTTY PAPER CUPS, SET/20...   
301  frozenset({SET/6 RED SPOTTY PAPER PLATES, SET/...   

                                    consequents  antecedent support  \
44                         frozenset({POSTAGE})            0.076531   
258                        frozenset({POSTAGE})            0.051020   
270                        frozenset({POSTAGE})            0.053571   
302  frozenset({SET/6 RED SPOTTY PAPER PLATES})            0.102041   
301    frozenset({SET/6 RED SPOTTY PAPER CUPS})            0.102041   

     consequent support   support  confidence      lift  representativity  \
44             0.765306  0.076531       1.000  1.306667               1.0   
258            0.765306  0.051020       1.000  1.306667               